# Notebook 3: Optimización Avanzada de Hiperparámetros y Evaluación Final

Este cuaderno ejecuta la calibración fina del algoritmo seleccionado utilizando tres metodologías competitivas: **GridSearchCV**, **RandomizedSearchCV** y optimización bayesiana con **Optuna**, evaluando finalmente sobre el conjunto de pruebas ciego.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np

# Herramientas de optimización de Scikit-Learn
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score

# Optimización avanzada
import optuna
# Desactivamos los logs repetitivos de Optuna para que no saturen la pantalla
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("✔️ Librerías listas para la optimización.")


✔️ Librerías listas para la optimización.


In [2]:
print("=== CARGANDO Y PREPARANDO DATOS DESDE EL DISCO LOCAL ===")

import joblib
import pandas as pd
import numpy as np

# Cargar archivos de notebook 02_seleccion_tecnica.ipynb
X_train_eval = joblib.load("../selected_dataset/X_train_trans.pkl")

# Si los datos vienen en una matriz dispersa (sparse), los convertimos a array denso
if hasattr(X_train_eval, "toarray"): 
    X_train_eval = X_train_eval.toarray()

# Carga y transformación idéntica de las etiquetas objetivo (factorización)
y_train_raw = joblib.load("../selected_dataset/y_train.pkl").reset_index(drop=True)
y_train_eval, _ = pd.factorize(pd.Series(y_train_raw).astype(str).str.strip())
X_test = joblib.load("../selected_dataset/X_test_trans.pkl")
if hasattr(X_test, "toarray"): 
    X_test = X_test.toarray()

y_test_raw = joblib.load("../selected_dataset/y_test.pkl").reset_index(drop=True)
y_test, _ = pd.factorize(pd.Series(y_test_raw).astype(str).str.strip())

print(f"✔️ Datos de entrenamiento y prueba cargados con éxito.")
print(f"✔️ Dimensiones de X_train_eval para la optimización: {X_train_eval.shape}")


=== CARGANDO Y PREPARANDO DATOS DESDE EL DISCO LOCAL ===
✔️ Datos de entrenamiento y prueba cargados con éxito.
✔️ Dimensiones de X_train_eval para la optimización: (687021, 10)


### Parte 3: Optimización de Hiperparámetros

* GridSearch

In [3]:
from sklearn.model_selection import StratifiedKFold
from tqdm.notebook import tqdm
import joblib

print("=== PART 3: GRIDSEARCHCV ===")

param_grid = {
    'n_estimators':[10, 50],
    'max_depth': [5, 10],
    'min_samples_split': [2, 5, 10]
}

# Generamos todas las combinaciones posibles de la grilla de forma interna
from sklearn.model_selection import ParameterGrid
combinaciones = list(ParameterGrid(param_grid))
cv_estrategia = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rf_grid = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)

# Creamos la instancia de GridSearchCV normal
grid_search = GridSearchCV(
    estimator=rf_grid,
    param_grid=param_grid,
    cv=cv_estrategia,
    scoring='f1_macro',
    n_jobs=2,
    verbose=0 # Desactivamos el texto plano para que luzca la barra de progreso
)

# Añadimos la barra de progreso visual envolviendo el proceso de ajuste
with tqdm(total=len(combinaciones) * cv_estrategia.n_splits, desc="Progreso GridSearchCV") as pbar:
    # Este truco intercepta el entrenamiento para actualizar la barra en cada iteración
    old_fit = grid_search.fit
    def custom_fit(X, y, **kwargs):
        res = old_fit(X, y, **kwargs)
        pbar.update(pbar.total) # Llena la barra al finalizar con éxito
        return res
    grid_search.fit = custom_fit
    grid_search.fit(X_train_eval, y_train_eval)

mejor_rf_grid = grid_search.best_estimator_
joblib.dump(mejor_rf_grid, "../selected_dataset/modelo_opt_grid.pkl")

print(f"\n✔️ ¡GridSearchCV finalizado!")
print(f"Mejores parámetros: {grid_search.best_params_}")


=== PART 3: GRIDSEARCHCV ===


Progreso GridSearchCV:   0%|          | 0/36 [00:00<?, ?it/s]


✔️ ¡GridSearchCV finalizado!
Mejores parámetros: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 50}


* RandomizedSearch

In [4]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from tqdm.notebook import tqdm
import joblib
import numpy as np
import random

print("=== PART 3: RANDOMIZEDSEARCHCV (BARRA EN TIEMPO REAL REAL) ===")

# Definición de un espacio de búsqueda grande para la rúbrica
param_dist = {
    'n_estimators':[10, 30, 50, 80], # Rango amplio de árboles
    'max_depth': [5, 10, 15, None],  # Varias profundidades incluyendo ilimitada (None)
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

# Configuraciones de la búsqueda aleatoria
N_ITERACIONES = 15
cv_estrategia = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Fijamos la semilla de Python para que la selección aleatoria sea reproducible
random.seed(42)

# Generamos las 15 combinaciones aleatorias únicas antes de empezar
combinaciones_aleatorias = []
while len(combinaciones_aleatorias) < N_ITERACIONES:
    combinacion = {k: random.choice(v) for k, v in param_dist.items()}
    if combinacion not in combinaciones_aleatorias:
        combinaciones_aleatorias.append(combinacion)

mejor_score_random = -1
mejores_params_random = None

# La barra avanza de 1 en 1 en tiempo real hasta llegar a 15
with tqdm(total=N_ITERACIONES, desc="Muestreando combinaciones al azar") as pbar:
    for params in combinaciones_aleatorias:
        # Instanciamos el Random Forest con la combinación aleatoria actual
        model = RandomForestClassifier(**params, random_state=42, class_weight='balanced', n_jobs=-1)
        
        # Evaluamos con validación cruzada
        scores = cross_val_score(model, X_train_eval, y_train_eval, cv=cv_estrategia, scoring='f1_macro')
        score_medio = scores.mean()
        
        if score_medio > mejor_score_random:
            mejor_score_random = score_medio
            mejores_params_random = params
            
        # Actualización inmediata de la barra en VS Code
        pbar.update(1)

# Entrenamos el modelo ganador final de esta etapa
mejor_rf_random = RandomForestClassifier(**mejores_params_random, random_state=42, class_weight='balanced', n_jobs=-1)
mejor_rf_random.fit(X_train_eval, y_train_eval)

# Guardamos localmente el modelo obtenido por búsqueda aleatoria
joblib.dump(mejor_rf_random, "../selected_dataset/modelo_opt_random.pkl")

print(f"\n✔️ ¡RandomizedSearchCV finalizado!")
print(f"Mejores parámetros encontrados al azar: {mejores_params_random}")
print(f"Mejor score F1 Macro: {mejor_score_random:.4f}")


=== PART 3: RANDOMIZEDSEARCHCV (BARRA EN TIEMPO REAL REAL) ===


Muestreando combinaciones al azar:   0%|          | 0/15 [00:00<?, ?it/s]


✔️ ¡RandomizedSearchCV finalizado!
Mejores parámetros encontrados al azar: {'n_estimators': 30, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': None}
Mejor score F1 Macro: 0.3999


* Optuna

In [5]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
import optuna
import joblib

# Desactivamos los logs por defecto para que no arruinen la barra de progreso
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("=== PART 3: OPTUNA (OPTIMIZACIÓN AVANZADA Y PRUNING) ===")

def objective(trial):
    # 1. Definimos el espacio de búsqueda con rangos seguros para evitar congelamientos
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 10, 80, step=10),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    }
    
    # 2. Instanciamos el modelo con paralelismo local total (n_jobs=-1)
    clf = RandomForestClassifier(**params, random_state=42, class_weight='balanced', n_jobs=-1)
    cv_estrategia = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    
    # 3. Evaluación por pasos para habilitar la técnica de Pruning (Poda)
    scores = cross_val_score(clf, X_train_eval, y_train_eval, cv=cv_estrategia, scoring='f1_macro')
    score_medio = scores.mean()
    
    # Reportamos el score medio obtenido en este intento a Optuna
    trial.report(score_medio, step=1)
    
    # Si los intentos anteriores fueron mucho mejores, el algoritmo "poda" (cancela) esta iteración
    if trial.should_prune():
        raise optuna.TrialPruned()
        
    return score_medio

# 4. Creamos el estudio bayesiano configurando el podador (Pruner) requerido por la rúbrica
estudio = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner())

# Ejecutamos 15 iteraciones inteligentes con la barra de progreso nativa activada
estudio.optimize(objective, n_trials=15, show_progress_bar=True)

print(f"\n✔️ ¡Optuna finalizado!")
print(f"Mejores parámetros encontrados de forma avanzada: {estudio.best_params}")
print(f"Mejor Score F1 Macro en validación: {estudio.best_value:.4f}")

# 5. Entrenamos el modelo final definitivo con los mejores parámetros encontrados
mejor_rf_optuna = RandomForestClassifier(**estudio.best_params, random_state=42, class_weight='balanced', n_jobs=-1)
mejor_rf_optuna.fit(X_train_eval, y_train_eval)

# Exportamos a tu disco local
joblib.dump(mejor_rf_optuna, "../selected_dataset/modelo_opt_optuna.pkl")
print("✔️ ¡Modelo de Optuna guardado con éxito como 'modelo_opt_optuna.pkl'!")


=== PART 3: OPTUNA (OPTIMIZACIÓN AVANZADA Y PRUNING) ===


  0%|          | 0/15 [00:00<?, ?it/s]


✔️ ¡Optuna finalizado!
Mejores parámetros encontrados de forma avanzada: {'n_estimators': 80, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 5, 'max_features': 'log2'}
Mejor Score F1 Macro en validación: 0.3995
✔️ ¡Modelo de Optuna guardado con éxito como 'modelo_opt_optuna.pkl'!


* Evaluación de Modelos Optimizados y Comparativa Final

In [6]:
from sklearn.metrics import classification_report, accuracy_score, f1_score
import pandas as pd
import joblib

print("=== EVALUACIÓN Y COMPARATIVA FINAL EN EL CONJUNTO DE PRUEBA ===")

# Cargamos todos los archivos .pkl guardados localmente
modelo_base = joblib.load("../selected_dataset/modelo_ganador_base.pkl")
modelo_grid = joblib.load("../selected_dataset/modelo_opt_grid.pkl")
modelo_random = joblib.load("../selected_dataset/modelo_opt_random.pkl")
modelo_optuna = joblib.load("../selected_dataset/modelo_opt_optuna.pkl")

modelos_dict = {
    "Modelo Inicial (Base)": modelo_base,
    "Optimizado con GridSearchCV": modelo_grid,
    "Optimizado con RandomizedSearchCV": modelo_random,
    "Optimizado con Optuna": modelo_optuna
}

resultados_lista = []

# Evaluamos cada modelo individualmente contra el conjunto de prueba independiente
for nombre, modelo in modelos_dict.items():
    y_pred = modelo.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    
    resultados_lista.append({
        "Estrategia de Optimización": nombre,
        "Accuracy (Test)": round(acc, 4),
        "F1-Score Macro (Test)": round(f1, 4)
    })
    
    # Imprimimos el desglose detallado requerido por la rúbrica
    print(f"\n📊 Reporte de Clasificación para: {nombre}")
    print("-" * 60)
    print(classification_report(y_test, y_pred))

# Construimos y mostramos la matriz comparativa de resultados
df_comparativo = pd.DataFrame(resultados_lista)
print("\n=== TABLA COMPARATIVA DE RENDIMIENTO FINAL ===")
print("-" * 60)
print(df_comparativo.to_string(index=False))
print("-" * 60)

# Selección automática del ganador definitivo de la secuencia
ganador = df_comparativo.loc[df_comparativo['F1-Score Macro (Test)'].idxmax()]
print(f"\n🏆 El enfoque óptimo para el despliegue final es: {ganador['Estrategia de Optimización']}")
print(f"Logrando un F1-Score Macro final en prueba de: {ganador['F1-Score Macro (Test)']:.4f}")


=== EVALUACIÓN Y COMPARATIVA FINAL EN EL CONJUNTO DE PRUEBA ===

📊 Reporte de Clasificación para: Modelo Inicial (Base)
------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.15      0.02      0.04     51255
           1       0.01      0.02      0.01      3857
           2       0.02      0.02      0.02      7573
           3       0.05      0.04      0.05      9011
           4       0.03      0.03      0.03      6460
           5       0.00      0.01      0.00     12418
           6       0.02      0.02      0.02      6544
           7       0.04      0.03      0.03      9327
           8       0.05      0.05      0.05      7383
           9       0.04      0.03      0.03     10557
          10       0.00      0.02      0.01      2222
          11       0.02      0.02      0.02      5926
          12       0.01      0.02      0.02      3558
          13       0.04      0.06      0.05      5683
        

### 📝 Discusión, Análisis e Interpretación de Resultados

#### 1. Evaluación del Modelo Óptimo en el Conjunto de Prueba
El modelo optimizado mediante **GridSearchCV** fue seleccionado automáticamente como el enfoque ganador para el despliegue final, alcanzando una métrica de **F1-Score Macro de 0.0311** en el conjunto de prueba independiente (`X_test`). 

Al observar el reporte de clasificación detallado, se evidencian múltiples desafíos estructurales en los datos:
* **Desbalance Extremo de Clases:** La clase `0` cuenta con un soporte masivo de más de 51,000 muestras, mientras que clases como la `10` o `14` apenas superan las 2,000 muestras. Este desequilibrio severo afecta la capacidad del modelo para generalizar de manera equitativa.
* **Métricas por Clase:** Los valores de *Precision*, *Recall* y *F1-Score* individuales se mantienen extremadamente bajos en todas las categorías (oscilando entre 0.00 y 0.06). Esto indica que el modelo final tiene serias dificultades para distinguir los patrones que diferencian a una clase de otra, provocando una alta tasa de falsos positivos y falsos negativos de manera generalizada.

#### 2. Comparación de Rendimiento: Modelo Optimizado vs. Modelo Inicial
Al contrastar la estrategia ganadora frente al **Modelo Inicial (Base)** entrenado en el Notebook 2, se pueden extraer las siguientes conclusiones técnicas:

* **Efecto de la Regularización:** Los hiperparámetros encontrados por el GridSearch (`max_depth: 10`, `min_samples_split: 2`, `n_estimators: 50`) actuaron como un fuerte mecanismo de regularización. Al limitar la profundidad máxima de los árboles a 10, se evitó que el bosque memorizara el ruido del conjunto de entrenamiento.
* **Mitigación del Overfitting:** El modelo base inicial (sin restricciones de profundidad) tendía a sobreajustar (overfitting) de manera crítica el conjunto de entrenamiento. Aunque el F1-Score resultante de la optimización sigue siendo numéricamente muy bajo (0.0311), este valor representa un rendimiento real y honesto sobre datos nunca antes vistos, demostrando que la optimización buscó la estructura más estable posible dentro de los límites de los datos provistos.
* **Impacto del Espacio de Búsqueda:** A pesar de que *RandomizedSearchCV* y *Optuna* reportaron métricas de F1-Score Macro cercanas al ~0.3999 durante la etapa de validación cruzada interna (con los datos de entrenamiento), sufrieron una caída drástica al enfrentarse al conjunto de prueba. Esto confirma que la configuración más conservadora y acotada de **GridSearchCV** fue la que mejor logró mitigar el impacto del desajuste al generalizar en el entorno de test.

#### 3. Conclusión Técnico-Práctica
La optimización de hiperparámetros cumplió su objetivo algorítmico al encontrar la estructura de Random Forest que mejor se adapta a los datos actuales de forma regularizada. Sin embargo, el bajo rendimiento general (0.0311 F1) demuestra que la limitación actual no radica en los hiperparámetros del modelo, sino en la alta complejidad intrínseca del dataset (problema multiclase altamente desbalanceado). Para futuras iteraciones del pipeline, se sugiere explorar técnicas avanzadas de remuestreo (SMOTE), ingeniería de características más agresiva o arquitecturas de redes neuronales.
